# SystemOne Guard — Train, Calibrate, Evaluate and Compare

**From scratch:** A small parallel CNN/attention model produces typed probabilities for `allow`, `review`, `block` and three auxiliary decisions. No Jev proprietary weights, no Jev RLCD, and no pretrained backbone.

**Important:** This is an educational **synthetic-policy** benchmark. The model misses **all ten** manually written novel prompt-injection phrasings. Never use it as a real security enforcement mechanism. The pretrained LLM comparator requires network access and has not been run for the packaged metrics.


## 1. Upload project ZIP (Colab only)

In [ ]:
import os
import sys
import zipfile
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src" / "systemone_guard").exists():
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError(
            "Run this notebook in the extracted project root, or upload the ZIP in Colab"
        ) from exc
    uploaded = files.upload()  # Select systemone-guard-complete.zip
    archive = next(name for name in uploaded if name.endswith(".zip"))
    with zipfile.ZipFile(archive) as z:
        z.extractall(".")
    ROOT = Path.cwd() / "systemone-guard"
    os.chdir(ROOT)
print("Project root:", Path.cwd())
print(
    "Pretrained checkpoint available:", Path("artifacts/guard_best/model.pt").exists()
)

## 2. Setup

In [ ]:
import importlib.util
import subprocess

if (
    importlib.util.find_spec("systemone_guard") is None
    or importlib.util.find_spec("pytest") is None
):
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-e", ".[dev,notebook]"]
    )
import json

import numpy as np
import torch

print("PyTorch:", torch.__version__, "CUDA:", torch.cuda.is_available())
torch.set_num_threads(2)

## 3. Inspect synthetic dataset and task balance

In [ ]:
from collections import Counter

from systemone_guard.data import CLASSES, load

rows = {split: load(split) for split in ("train", "val", "test", "challenge")}
for split, data in rows.items():
    print(split, len(data), dict(Counter(x["decision"] for x in data)))
print(json.dumps(rows["test"][0], indent=2))

## 4. Reproduce model training (optional: off by default)

In [ ]:
RUN_FULL_TRAINING = (
    False  # True trains from random initialization and saves another checkpoint
)
if RUN_FULL_TRAINING:
    from systemone_guard.train import train

    new_run = train(
        data="data",
        out="artifacts/notebook_retrained",
        seed=7,
        epochs=6,
        dim=32,
        hidden=64,
        lr=0.0015,
        batch=128,
    )
    print(new_run["best_val_macro_f1"])
else:
    print(
        "Using provided fully trained checkpoint. Toggle RUN_FULL_TRAINING=True to retrain."
    )

## 5. One-pass probabilistic inference

In [ ]:
from systemone_guard.inference import DecisionEngine

engine = DecisionEngine("artifacts/guard_best/model.pt")
example = dict(
    request="Please read customer records for an incident review",
    context="Tool returned a normal response.",
    action="read",
    sensitivity="internal",
    destination="internal",
    authorization="approved",
    source="user",
)
print(json.dumps(engine.decide(**example), indent=2))

## 6. Evaluate all saved models

In [ ]:
from systemone_guard.evaluate import eval_split

RUN_FULL_EVALUATION = (
    False  # Set True to recompute all saved metrics and per-example predictions
)
for split in ["val", "test", "challenge"]:
    if RUN_FULL_EVALUATION:
        results = eval_split(split, include_causal=True)
    else:
        results = json.loads(Path("experiments", split + "_metrics.json").read_text())
    print(
        split, {name: round(scores["accuracy"], 4) for name, scores in results.items()}
    )

## 7. Generate plots from actual run histories and evaluation files

In [ ]:
subprocess.check_call([sys.executable, "scripts/plots.py", "results"])
from IPython.display import Image, display

for file in [
    "comparison.png",
    "train_loss.png",
    "val_f1.png",
    "val_loss.png",
    "calibration.png",
    "confusion.png",
]:
    display(Image(filename=str(Path("figures") / file)))

## 8. Manual independent adversarial probes (crucial limitation)

In [ ]:
subprocess.check_call([sys.executable, "scripts/evaluate_probes.py"])
probe = json.loads(Path("experiments/manual_probes.json").read_text())
print("Probe accuracy:", probe["accuracy"])
print("By group:", json.dumps(probe["by_group"], indent=2))

## 9. Run tests and CPU batch-latency benchmark

In [ ]:
subprocess.check_call([sys.executable, "-m", "pytest", "-q", "tests"])
subprocess.check_call([sys.executable, "scripts/benchmark_latency.py"])

    ## 10. Optional: benchmark a real pretrained System Two LLM on the SAME examples

Not included in published numerical results. Requires internet to download Hugging Face weights; use CPU for both methods. Please record model revision and hardware.

In [ ]:
RUN_PRETRAINED_LLM_COMPARISON = False
if RUN_PRETRAINED_LLM_COMPARISON:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".[llm]"])
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "systemone_guard.benchmarking.pretrained",
            "--models",
            "HuggingFaceTB/SmolLM2-135M-Instruct",
            "--split",
            "test",
            "--count",
            "200",
            "--threads",
            "2",
            "--device",
            "cpu",
        ]
    )
else:
    print("Pretrained LLM benchmark not run; no unverified results reported.")

    ## 11. Hugging Face publication

The provided checkpoint uses custom PyTorch source (`systemone_guard`). Upload the complete source, model card and weights, then deploy `spaces/app.py` in a Space with required dependencies. Do not represent these weights as Jev or production-grade security detection.

In [ ]:
from pathlib import Path

for p in [
    "hf_model/README.md",
    "artifacts/guard_best/model.pt",
    "spaces/app.py",
    "experiments/test_metrics.json",
]:
    print(p, Path(p).exists())